<a href="https://www.kaggle.com/code/jayhawk1900/f1-realmlp?scriptVersionId=321410940" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# F1 Pit Stop Prediction: RealMLP (Neural Network for Tabular Data)

**Competition:** [Playground Series S6E5 — Predicting F1 Pit Stops](https://www.kaggle.com/competitions/playground-series-s6e5)

**Metric:** ROC-AUC | **Task:** Binary classification — will this driver pit on the next lap?

---

## Why a Neural Network?

Gradient-boosted decision trees (XGBoost, CatBoost, LightGBM) usually dominate tabular competitions, and a tuned GBDT blend is a strong baseline. But blending models from the *same family* hits a ceiling — they make correlated errors.

This notebook implements **RealMLP**, a neural architecture (from the [PyTabKit](https://github.com/dholzmueller/pytabkit) research library) engineered specifically to make MLPs competitive with GBDTs on tabular data. It does this through three key ideas:

1. **PBLD embeddings** — encode numeric features as periodic (sine/cosine) waves at learned frequencies, letting the network create sharp, threshold-like responses similar to tree splits.
2. **Internal ensemble** — train 16 networks in parallel inside one model and average them, reducing variance.
3. **A carefully scheduled training recipe** — per-parameter-group learning rates, label-smoothing and dropout schedules, and class weighting for imbalance.

Because a neural network "sees" the data completely differently from trees, its predictions are **uncorrelated** with GBDTs — making it an ideal blend partner. A single RealMLP can match or beat an entire GBDT ensemble, and blending the two beats either alone.

---

## Approach

- Feature engineering: arithmetic interactions, numeric-as-categorical, count encoding, binning
- The original F1 strategy dataset concatenated as additional training data (a strong signal on this synthetic-but-ORIG-derived data)
- Cross-validated target encoding on key categorical interactions
- 5-fold stratified CV, GPU training (~25 min)

## Step 1: Feature Engineering

Neural networks benefit from richer input representations than raw features. We create several types of engineered features (this pipeline is shared with the GBDT models too):

**Arithmetic interactions** — ratios and products that capture domain relationships:
- `LapNumber / RaceProgress` — pace through the race
- `TyreLife / LapNumber` — how worn the tires are relative to race stage
- `LapTime × Cumulative_Degradation` — compounded time loss

**Numeric-as-categorical** — floor each numeric value and treat it as a category. This lets the model's embedding layers learn per-bucket effects (e.g. "lap 23 specifically behaves like X"), capturing non-linear jumps that raw numerics smear over.

**Count encoding** — replace each category with how often it appears. Frequency itself is informative (common driver/compound combos behave differently from rare ones).

**Binning** — `KBinsDiscretizer` splits continuous features (RaceProgress, LapTime) into quantile bins, another way to expose threshold effects.

**Interaction categories** — combine `Race+Compound` and `Race+Year` into single categorical keys for target encoding later.

The same `feature_engineering` function is fit on train and applied to test and the ORIG dataset, so all three share identical feature definitions.

In [1]:
import math, random, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')

def seed_everything(seed):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

# Load data
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'
ORIG_DIR = '/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
orig = pd.read_csv(f'{ORIG_DIR}/f1_strategy_dataset_v4.csv')
print('Loaded:', train.shape, test.shape, orig.shape)

ID, TARGET = 'id', 'PitNextLap'
orig = orig.drop(['Normalized_TyreLife'], axis=1)
y_orig = orig[TARGET]; orig = orig.drop([TARGET], axis=1)
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print('cat_cols:', cat_cols)
print('num_cols:', len(num_cols))

category_map = {}
important_combos = [('Race', 'Compound'), ('Race', 'Year')]

def feature_engineering(df, fit=False):
    df = df.copy()
    # Arithmetic interactions
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')

    # Categorize string categoricals
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Numeric-as-categorical (floor + factorize)
    for col in num_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']:
        cat_name = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Count encoding
    for col in cat_cols + ['Year_cat_', 'PitStop_cat_']:
        count_name = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"
        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].astype(object).map(count_map).fillna(0).astype('int32')

    # Discretize numericals (quantile bins)
    bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype('int32')
            df[bin_name] = binned
            df[bin_name] = df[bin_name].astype('category')

    # Interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for c in cols[1:]:
            combo_series = combo_series + '_' + df[c].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [c for c in df.columns if c.endswith('_')]
    new_num_cols = [c for c in df.columns if c.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
orig, _, _, _ = feature_engineering(orig, fit=False)

cat_cols_all = cat_cols + new_cat_cols
num_cols_all = num_cols + new_num_cols
print(f'\nAfter FE: {X.shape}')
print(f'Total cat cols: {len(cat_cols_all)}, num cols: {len(num_cols_all)}')
print(f'Combo names (for target encoding): {combo_names}')

PyTorch: 2.10.0+cu128 | device: cuda
Loaded: (439140, 16) (188165, 15) (101371, 16)
cat_cols: ['Driver', 'Compound', 'Race']
num_cols: 11

After FE: (439140, 41)
Total cat cols: 20, num cols: 21
Combo names (for target encoding): ['Race_Compound_', 'Race_Year_']


## Step 2: PBLD Embedding — Periodic Encoding for Numbers

This is the key idea that makes the MLP competitive with trees.

**The problem:** Feeding a raw number like `TyreLife = 23` into a neural network is hard for it to use. Networks struggle to learn sharp, threshold-like responses from raw numerics — exactly the kind of "if value > X" logic that decision trees do effortlessly.

**The solution:** PBLD (Periodic Basis with Learned Decay) transforms each number into a richer vector using **cosine waves at learned frequencies**. For a single input value, it computes:

`cos(2π · (x · frequency + phase))`

across many learned frequency/phase pairs. The network learns *which* frequencies are useful during training. This is conceptually similar to the positional encodings used in transformers.

**Why it works:** Different frequencies let the network detect patterns at different scales, and combinations of periodic features can approximate the sharp decision boundaries that trees create naturally. The raw feature is also kept (a residual connection) so no information is lost.

The implementation below is fully vectorized — it processes all numeric features and all 16 ensemble members in parallel using `einsum`, with no Python loops.

In [2]:
class PBLDEmbedding(nn.Module):
    """Periodic Basis with Learned Decay embedding for numerical features.

    For each numeric feature, produces `out_dim` values:
      - the raw feature (residual), plus
      - (out_dim - 1) learned periodic transforms.
    Vectorized over n_ens ensemble members and all features.
    """
    def __init__(self, n_ens, n_features, hidden_dim=16, out_dim=4,
                 freq_scale=0.1, activation=nn.GELU):
        super().__init__()
        self.n_ens = n_ens
        self.n_features = n_features
        self.out_dim = out_dim
        # Learned frequencies (w1) and phases (b1) for the periodic basis
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        # Projects the periodic basis down to (out_dim - 1) values
        self.w2 = nn.Parameter(
            torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim)
        )
        self.b2 = nn.Parameter(torch.randn(n_ens, n_features, out_dim - 1))
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)  # phases spread across a full cycle

    def forward(self, x):
        # x: (batch, n_ens, n_features)
        # Build periodic basis: cos(2π (x·freq + phase)) → (batch, n_ens, n_features, hidden)
        periodic = torch.cos(
            2 * math.pi * (
                x.unsqueeze(-1) * self.w1.unsqueeze(0)
                + self.b1.unsqueeze(0)
            )
        )
        # Project down and activate → (batch, n_ens, n_features, out_dim-1)
        transformed = self.act(
            torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2)
            + self.b2.unsqueeze(0)
        )
        # Keep the raw feature (residual) alongside the periodic transforms
        feat = torch.cat([x.unsqueeze(-1), transformed], dim=-1)  # (batch, n_ens, n_features, out_dim)
        return feat.flatten(start_dim=2)  # (batch, n_ens, n_features * out_dim)

# Quick sanity test on random data
_test = PBLDEmbedding(n_ens=2, n_features=3, hidden_dim=8, out_dim=4).to(device)
_x = torch.randn(5, 2, 3).to(device)  # (batch=5, n_ens=2, features=3)
_out = _test(_x)
print('PBLD input shape: ', tuple(_x.shape))
print('PBLD output shape:', tuple(_out.shape), '= (batch, n_ens, features × out_dim)')
print('Expected output last dim: 3 features × 4 out_dim =', 3*4)
del _test, _x, _out

PBLD input shape:  (5, 2, 3)
PBLD output shape: (5, 2, 12) = (batch, n_ens, features × out_dim)
Expected output last dim: 3 features × 4 out_dim = 12


## Step 3: Categorical, Scaling, and Ensemble-Aware Layers

Three supporting components:

**CategoricalFeatureLayer** — handles categories two ways depending on cardinality:
- **Low-cardinality** (≤ `onehot_thresh`, e.g. Compound with 5 values) → one-hot encoded
- **High-cardinality** (e.g. Driver with 887 values) → learned embeddings (dense vectors, one per ensemble member)

This avoids exploding the input width for high-cardinality features while still giving the model a one-hot signal where it's cheap.

**ScalingLayer** — a simple learned per-feature scale applied before the MLP. It lets the network learn how much to amplify or dampen each input dimension, a small but helpful addition.

**NTPLinear** — the workhorse linear layer, but with a twist: it carries an `n_ens` dimension so all 16 ensemble members compute in parallel via a single `einsum`. This is **Trick 2 (the internal ensemble)** in action — 16 networks for far less than 16× the cost. The `/ sqrt(in_features)` is a normalization (Neural Tangent Parameterization) that stabilizes training.

In [3]:
class CategoricalFeatureLayer(nn.Module):
    """One-hot for low-cardinality cats, learned embeddings for high-cardinality."""
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []
        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                emb = nn.ModuleList([nn.Embedding(dim, embed_dim) for _ in range(n_ens)])
                self.embed_layers.append(emb)
                self._embed_feature_indices.append(i)

    def forward(self, x):
        # x: (batch, n_ens, n_cat)
        batch_size, n_ens, _ = x.shape
        features = []
        if self.onehot_features:
            onehot_x = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_oh = sum(onehot_dims)
            encoded = torch.zeros(batch_size, n_ens, total_oh, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx:idx+1].long()
                encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(encoded)
        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = []
            for model_idx in range(self.n_ens):
                indices = x[:, model_idx, feat_idx:feat_idx+1].long()
                feat_embs.append(emb_list[model_idx](indices))
            feat_combined = torch.cat(feat_embs, dim=1)
            features.append(feat_combined)
        return torch.cat(features, dim=2)


class ScalingLayer(nn.Module):
    """Learned per-feature scale, per ensemble member."""
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))
    def forward(self, x):
        return x * self.scale[None, :, :]


class NTPLinear(nn.Module):
    """Linear layer with an ensemble dimension; all n_ens members in one einsum."""
    def __init__(self, n_ens, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None
    def forward(self, x):
        # x: (batch, n_ens, in_features) → (batch, n_ens, out_features)
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x

# Sanity test
_cat = CategoricalFeatureLayer(n_ens=2, cat_dims=[3, 887, 5], embed_dim=6, onehot_thresh=4).to(device)
_xc = torch.randint(0, 3, (4, 2, 3)).to(device)  # 4 rows, 2 ens, 3 cat features
_oc = _cat(_xc)
print('Cat layer output shape:', tuple(_oc.shape))
print('  (cat_dims [3,887,5], thresh=4: dims 3 and 5 → one-hot=8, dim 887 → embed=6; total 14)')

_lin = NTPLinear(n_ens=2, in_features=10, out_features=7).to(device)
_xl = torch.randn(4, 2, 10).to(device)
print('NTPLinear output shape:', tuple(_lin(_xl).shape), '= (batch, n_ens, out_features)')
del _cat, _xc, _oc, _lin, _xl

Cat layer output shape: (4, 2, 15)
  (cat_dims [3,887,5], thresh=4: dims 3 and 5 → one-hot=8, dim 887 → embed=6; total 14)
NTPLinear output shape: (4, 2, 7) = (batch, n_ens, out_features)


## Step 4: The RealMLP Model

Now we assemble the components into the full network. The forward pass:

1. **Numeric features** → PBLD embedding (periodic expansion)
2. **Categorical features** → CategoricalFeatureLayer (one-hot + embeddings)
3. **Concatenate** both into one wide feature vector
4. Optional **ScalingLayer** at the front
5. Pass through the **hidden MLP** (`[512, 64, 128]` with SiLU activation + dropout)
6. **Output layer** → sigmoid → P(pit)

Every layer carries the `n_ens=16` dimension, so 16 networks run in parallel. At inference, their 16 outputs are averaged.

The architecture tracks the **first hidden linear layer** separately, because the training recipe (next step) gives it its own learning rate. Each dropout layer is a distinct instance so the dropout schedule can update them individually.

In [4]:
class RealMLP(nn.Module):
    def __init__(self, output_dim, cat_dims, n_numerical, cfg):
        super().__init__()
        n_ens = cfg["n_ens"]
        embed_dim = cfg["embed_dim"]
        self.n_ens = n_ens

        self.cate = CategoricalFeatureLayer(
            n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim,
            onehot_thresh=cfg["onehot_thresh"],
        )
        self.num_embed = PBLDEmbedding(
            n_ens=n_ens, n_features=n_numerical,
            hidden_dim=cfg["pbld_hidden_dim"], out_dim=cfg["pbld_out_dim"],
            freq_scale=cfg["pbld_freq_scale"], activation=cfg["pbld_activation"],
        )

        num_emb_dim = n_numerical * cfg["pbld_out_dim"]
        cat_emb_dim = sum(c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total_dim = num_emb_dim + cat_emb_dim
        hidden_dims = cfg["hidden_dims"]
        act = cfg["activation"]

        layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens=n_ens, n_features=total_dim))

        self._dropout_modules = []
        in_dim = total_dim
        for i, out_dim_h in enumerate(hidden_dims):
            linear = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=out_dim_h)
            if i == 0:
                self.first_linear = linear  # tracked for its own lr group
            drop = nn.Dropout(cfg["dropout"])
            self._dropout_modules.append(drop)
            layers += [linear, act(), drop]
            in_dim = out_dim_h

        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        combined = torch.cat([x_num, x_cat], dim=2)
        x = self.hidden(combined)
        x = self.output_layer(x)
        return torch.sigmoid(x)  # (batch, n_ens, 1)

# Sanity test with a small config
_cfg_test = {
    "n_ens": 4, "embed_dim": 6, "onehot_thresh": 4,
    "hidden_dims": [64, 32], "dropout": 0.05, "activation": nn.SiLU,
    "add_front_scale": True,
    "pbld_hidden_dim": 16, "pbld_out_dim": 4, "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU,
}
_model = RealMLP(output_dim=1, cat_dims=[3, 887, 5], n_numerical=21, cfg=_cfg_test).to(device)
_xn = torch.randn(8, 21).to(device)
_xc = torch.randint(0, 3, (8, 3)).to(device)
_out = _model(_xn, _xc)
print('Model output shape:', tuple(_out.shape), '= (batch, n_ens, 1)')
print('Averaged over ensemble → per-row prediction:', tuple(_out.mean(dim=1).squeeze(-1).shape))
n_params = sum(p.numel() for p in _model.parameters())
print(f'Param count (small test config): {n_params:,}')
del _model, _xn, _xc, _out

Model output shape: (8, 4, 1) = (batch, n_ens, 1)
Averaged over ensemble → per-row prediction: (8,)
Param count (small test config): 62,829


## Step 5: The Training Recipe — Schedules, Loss, Parameter Groups

A vanilla training loop wouldn't get an MLP to GBDT-level performance. RealMLP uses a carefully tuned recipe:

**Schedules** (`apply_schedule`) — several values change over the course of training based on `progress` (0 → 1):
- **Learning rate**: `flat_cos` — held flat for the first 30%, then cosine-decayed to 0
- **Label smoothing**: starts at 0.04, cosine-decays to 0 — prevents early overconfidence
- **Dropout**: `expm4t` exponential decay — heavy regularization early, loosening as training settles

**Loss** (`binary_bce_loss`) — binary cross-entropy with two additions:
- **Label smoothing** — softens the 0/1 targets
- **pos_weight** — up-weights the positive (pit) class to handle the 80/20 imbalance, derived from balanced class weights

**Parameter groups** (`get_parameter_groups`) — five groups, each with its own learning rate:
- Scaling layer (high lr ×10)
- PBLD embeddings (×0.093 — they learn slower)
- First hidden layer weights
- All other weights (base lr)
- Biases (×0.1)

This per-group tuning is a big part of why the architecture trains well — different parts of the network need very different learning rates.

In [5]:
def apply_schedule(init_value, progress, sched, flat_ratio=0.3):
    """Schedules a value over training progress (0→1)."""
    if sched == "constant":
        return init_value
    elif sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    elif sched == "flat_cos":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (math.cos(math.pi * t) + 1) / 2
    elif sched == "flat_anneal":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - t)
    elif sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    elif sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    else:
        raise ValueError(f"Unknown schedule: '{sched}'")


def get_parameter_groups(model, p):
    """Five param groups, each with its own lr / weight decay."""
    first_linear_weight_id = id(model.first_linear.weight)
    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)
    LR = p["lr"]; WD = p["weight_decay"]
    return [
        {"params": scale_p,   "lr": LR * p["lr_scale_mult"],         "weight_decay": WD * p["wd_scale_mult"], "group": "scale"},
        {"params": pbld_p,    "lr": LR * p["pbld_lr_factor"],        "weight_decay": WD,                      "group": "pbld"},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay": WD,                      "group": "first_w"},
        {"params": other_w_p, "lr": LR,                              "weight_decay": WD,                      "group": "other_w"},
        {"params": bias_p,    "lr": LR * p["lr_bias_mult"],          "weight_decay": WD * p["wd_bias_mult"],  "group": "bias"},
    ]


def binary_bce_loss(y_true, y_pred, ls=0.0, pos_weight=None, eps=1e-7):
    """BCE with optional label smoothing and positive-class weighting."""
    if ls > 0.0:
        y_true = y_true * (1.0 - ls) + 0.5 * ls
    y_pred = y_pred.clamp(eps, 1.0 - eps)
    if pos_weight is None:
        loss = -y_true * torch.log(y_pred) - (1.0 - y_true) * torch.log(1.0 - y_pred)
    else:
        loss = -pos_weight * y_true * torch.log(y_pred) - (1.0 - y_true) * torch.log(1.0 - y_pred)
    return loss.mean()

# Sanity test the schedules
print('Schedule values at progress 0.0, 0.5, 1.0:')
for s in ['flat_cos', 'cos', 'expm4t']:
    vals = [round(apply_schedule(1.0, prog, s), 4) for prog in [0.0, 0.5, 1.0]]
    print(f'  {s:10s}: {vals}')

# Sanity test the loss
_yt = torch.tensor([1.0, 0.0, 1.0, 0.0]).to(device)
_yp = torch.tensor([0.9, 0.1, 0.6, 0.3]).to(device)
print(f'\nBCE loss (no weight):   {binary_bce_loss(_yt, _yp).item():.4f}')
print(f'BCE loss (ls=0.04):     {binary_bce_loss(_yt, _yp, ls=0.04).item():.4f}')
print(f'BCE loss (pos_weight=4):{binary_bce_loss(_yt, _yp, pos_weight=torch.tensor(4.0).to(device)).item():.4f}')
del _yt, _yp

Schedule values at progress 0.0, 0.5, 1.0:
  flat_cos  : [1.0, 0.8117, 0.0]
  cos       : [1.0, 0.5, 0.0]
  expm4t    : [1.0, 0.1353, 0.0183]

BCE loss (no weight):   0.2696
BCE loss (ls=0.04):     0.2978
BCE loss (pos_weight=4):0.7317


## Step 6: Training Wrapper (fit / predict)

A scikit-learn-style wrapper that ties the recipe together. Its `fit` method:

1. Splits features into numeric and categorical
2. Applies numeric preprocessing (median-center, robust-scale, smooth-clip)
3. Computes balanced class weights for the imbalance
4. Builds the RealMLP and the 5-group optimizer
5. Runs the epoch loop: for each batch, updates the lr/label-smoothing/dropout via the schedules, computes the weighted loss, backprops with gradient clipping
6. After each epoch, evaluates validation AUC and checkpoints the best model
7. Restores the best weights at the end

`predict_proba` averages the 16 ensemble members and returns calibrated probabilities. Numeric preprocessing and categorical clamping (guarding against unseen categories) mirror the fit step.

In [6]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    """Median-center, robust-scale, smooth-clip numeric features."""
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]
    def fit(self, X, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X


class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kwargs):
        self.params = {**CONFIG, **kwargs}

    def fit(self, X_train, y_train, X_val, y_val, cat_col_names=None,
            ckpt_path="realmlp_ckpt.pth", X_test=None):
        p = self.params
        dev = torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        verbose = p["verbosity"]
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        X_tr_num = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr = np.asarray(y_train); y_v = np.asarray(y_val)

        self.preprocessor_ = NumericalPreprocessor(p["tfms"])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names
        if cat_col_names:
            all_cat = [X_tr_cat, X_val_cat]
            if X_test is not None:
                all_cat.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims = (np.concatenate(all_cat, axis=0).max(axis=0) + 1).tolist()
        else:
            cat_dims = []
        self.cat_dims_ = cat_dims
        if cat_dims:
            cat_max = np.array(cat_dims) - 1
            X_tr_cat = np.clip(X_tr_cat, 0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        classes = np.unique(y_tr); self.classes_ = classes
        weights_np = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        pos_weight = torch.tensor(weights_np[1], dtype=torch.float32, device=dev)

        self.model_ = RealMLP(output_dim=1, cat_dims=cat_dims,
                              n_numerical=X_tr_num.shape[1], cfg=p).to(dev)
        param_groups = get_parameter_groups(self.model_, p)
        for g in param_groups:
            g["lr_base"] = g["lr"]
        optimizer = torch.optim.AdamW(param_groups, betas=(p["mom"], p["sq_mom"]))

        Xtn = torch.as_tensor(X_tr_num, dtype=torch.float32, device=dev)
        Xtc = torch.as_tensor(X_tr_cat, dtype=torch.long, device=dev)
        ytt = torch.as_tensor(y_tr, dtype=torch.float32, device=dev)
        Xvn = torch.as_tensor(X_val_num, dtype=torch.float32, device=dev)
        Xvc = torch.as_tensor(X_val_cat, dtype=torch.long, device=dev)

        n_ens = p["n_ens"]; train_bs = p["train_bs"]; eval_bs = p["eval_bs"]
        epochs = p["epochs"]; lr_sched = p["lr_sched"]; flat_ratio = p["flat_ratio"]
        total_steps = epochs * len(y_tr)
        train_order = np.arange(len(y_tr))
        best_score = -np.inf; best_epoch = 0; best_val_probs = None
        self.ckpt_path_ = ckpt_path

        for epoch in range(epochs):
            self.model_.train()
            for start in range(0, len(y_tr), train_bs):
                progress = (epoch * len(y_tr) + start) / total_steps
                idx_batch = train_order[start:start + train_bs]
                for g in optimizer.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, lr_sched, flat_ratio)
                optimizer.zero_grad()
                y_pred = self.model_(Xtn[idx_batch], Xtc[idx_batch])
                ls_val = apply_schedule(p["ls_eps"], progress, p["ls_eps_sched"], flat_ratio)
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"], flat_ratio)
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val
                loss = binary_bce_loss(
                    ytt[idx_batch].repeat_interleave(n_ens),
                    y_pred.reshape(-1), ls=ls_val, pos_weight=pos_weight,
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                optimizer.step()
            np.random.shuffle(train_order)

            self.model_.eval()
            with torch.no_grad():
                val_probs_pos = np.concatenate([
                    self.model_(Xvn[s:s+eval_bs], Xvc[s:s+eval_bs]).mean(dim=1).squeeze(-1).cpu().numpy()
                    for s in range(0, len(y_v), eval_bs)
                ], axis=0)
            epoch_score = roc_auc_score(y_v, val_probs_pos)
            improved = epoch_score > best_score
            if improved:
                best_score = epoch_score; best_epoch = epoch + 1
                best_val_probs = val_probs_pos.copy()
                torch.save(self.model_.state_dict(), ckpt_path)
            if verbose >= 2:
                print(f"  epoch {epoch+1}/{epochs}  score={epoch_score:.5f}  best={best_score:.5f}"
                      f"  ls={ls_val:.4f} drop={drop_val:.4f}" + (" *" if improved else ""))

        self.model_.load_state_dict(torch.load(ckpt_path))
        self.best_score_ = best_score
        self.best_val_probs_ = best_val_probs
        self._dev = dev
        if verbose >= 1:
            print(f"  -> best score: {best_score:.5f} (epoch {best_epoch})")
        return self

    def predict_proba(self, X):
        eval_bs = self.params["eval_bs"]
        X_num = self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        X_cat = X[self.cat_col_names_].values.astype(np.int64)
        X_cat = np.clip(X_cat, 0, np.array(self.cat_dims_) - 1)
        Xn = torch.as_tensor(X_num, dtype=torch.float32, device=self._dev)
        Xc = torch.as_tensor(X_cat, dtype=torch.long, device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            probs_pos = np.concatenate([
                self.model_(Xn[s:s+eval_bs], Xc[s:s+eval_bs]).mean(dim=1).squeeze(-1).cpu().numpy()
                for s in range(0, len(X_num), eval_bs)
            ], axis=0)
        return np.stack([1.0 - probs_pos, probs_pos], axis=1)

print('RealMLP_TD_Classifier defined.')

RealMLP_TD_Classifier defined.


## Step 7: Configuration and K-Fold Training

The CONFIG dict holds all hyperparameters — architecture, optimizer, schedules, preprocessing, and training loop settings. These match the reference RealMLP setup.

The training loop:
1. For each of 5 folds, **concatenate the ORIG dataset** into the training portion (the same +data lever that boosted our GBDTs)
2. Apply **cross-validated target encoding** on the `Race_Compound` and `Race_Year` interaction categories (fit inside each fold to prevent leakage)
3. Train a RealMLP, collect out-of-fold validation predictions and averaged test predictions

The validation fold is always pure competition data, so the OOF score is directly comparable to our GBDT models. With `n_ens=16` and 4 epochs, expect ~20-25 minutes on GPU.

In [7]:
CONFIG = {
    # Architecture
    "n_ens": 16, "embed_dim": 6, "onehot_thresh": 4,
    "hidden_dims": [512, 64, 128], "dropout": 0.05,
    "p_drop_sched": "expm4t", "activation": nn.SiLU, "add_front_scale": True,
    # PBLD embedding
    "pbld_hidden_dim": 20, "pbld_out_dim": 5, "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU, "pbld_lr_factor": 0.093,
    # Optimizer
    "lr": 0.008, "mom": 0.9, "sq_mom": 0.98, "lr_sched": "flat_cos",
    "flat_ratio": 0.3, "first_layer_lr_factor": 1.0, "lr_scale_mult": 10.0,
    "lr_bias_mult": 0.1, "weight_decay": 0.005, "wd_scale_mult": 0.1,
    "wd_bias_mult": 0.5, "grad_clip": 1.0,
    # Label smoothing
    "ls_eps": 0.04, "ls_eps_sched": "cos",
    # Preprocessing
    "tfms": ["median_center", "robust_scale", "smooth_clip"],
    # Training
    "epochs": 8, "train_bs": 256, "eval_bs": 10240, "verbosity": 2,
    "use_early_stopping": False,
    "early_stopping_additive_patience": 10,
    "early_stopping_multiplicative_patience": 1,
    # Device
    "device": "cuda", "random_state": 42,
}
print('CONFIG set. n_ens =', CONFIG['n_ens'], '| epochs =', CONFIG['epochs'])
print('cat_cols_all:', len(cat_cols_all), '| combos:', combo_names)

CONFIG set. n_ens = 16 | epochs = 8
cat_cols_all: 20 | combos: ['Race_Compound_', 'Race_Year_']


In [8]:
import time
FOLDS = 5
SEED = 42

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

t_start = time.time()
for fold, ((tr_idx, val_idx), (or_tr_idx, _)) in enumerate(
        zip(skf.split(X, y), skf.split(orig, y_orig)), 1):
    # Concatenate ORIG into the training fold
    X_tr = pd.concat([X.iloc[tr_idx], orig.iloc[or_tr_idx]], axis=0).reset_index(drop=True)
    y_tr = pd.concat([y.iloc[tr_idx], y_orig.iloc[or_tr_idx]], axis=0).reset_index(drop=True)
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx]
    X_tst = X_test.copy()

    # Cross-validated target encoding on combo categoricals (fit within fold)
    te = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED)
    tr_enc = te.fit_transform(X_tr[combo_names], y_tr)
    val_enc = te.transform(X_val[combo_names])
    tst_enc = te.transform(X_tst[combo_names])
    te_names = [f"_{c}TE" for c in combo_names]
    X_tr[te_names] = tr_enc
    X_val[te_names] = val_enc
    X_tst[te_names] = tst_enc

    if fold == 1:
        print('Total features:', len(X_tr.columns), '\n')
    print('#' * 16)
    print(f'### Fold {fold}/{FOLDS}')
    print('#' * 16)

    model = RealMLP_TD_Classifier(**CONFIG)
    model.fit(X_tr, y_tr, X_val, y_val,
              cat_col_names=cat_cols_all, ckpt_path=f'model_fold{fold}.pth')

    val_preds = model.best_val_probs_
    fold_test_preds = model.predict_proba(X_tst)[:, 1]
    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / FOLDS
    print(f'Fold {fold} | Score: {roc_auc_score(y_val, val_preds):.5f}\n')
    torch.cuda.empty_cache()

oof_auc = roc_auc_score(y, oof_preds)
print('=' * 26)
print(f'RealMLP OOF Score: {oof_auc:.5f}')
print('=' * 26)

# Save for blending with GBDTs
np.save('/kaggle/working/oof_realmlp.npy', oof_preds)
np.save('/kaggle/working/pred_realmlp.npy', test_preds)

# Build submission
sub = pd.DataFrame({ID: test_id, TARGET: test_preds})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved oof_realmlp.npy, pred_realmlp.npy, submission.csv')
print(f'Test pred mean: {test_preds.mean():.5f} (target ~0.199)')

Total features: 43 

################
### Fold 1/5
################
  epoch 1/8  score=0.92727  best=0.92727  ls=0.0385 drop=0.0303 *
  epoch 2/8  score=0.95233  best=0.95233  ls=0.0341 drop=0.0184 *
  epoch 3/8  score=0.95339  best=0.95339  ls=0.0277 drop=0.0112 *
  epoch 4/8  score=0.95384  best=0.95384  ls=0.0200 drop=0.0068 *
  epoch 5/8  score=0.95431  best=0.95431  ls=0.0123 drop=0.0041 *
  epoch 6/8  score=0.95415  best=0.95431  ls=0.0059 drop=0.0025
  epoch 7/8  score=0.95377  best=0.95431  ls=0.0015 drop=0.0015
  epoch 8/8  score=0.95353  best=0.95431  ls=0.0000 drop=0.0009
  -> best score: 0.95431 (epoch 5)
Fold 1 | Score: 0.95431

################
### Fold 2/5
################
  epoch 1/8  score=0.92212  best=0.92212  ls=0.0385 drop=0.0303 *
  epoch 2/8  score=0.94969  best=0.94969  ls=0.0341 drop=0.0184 *
  epoch 3/8  score=0.95160  best=0.95160  ls=0.0277 drop=0.0112 *
  epoch 4/8  score=0.95159  best=0.95160  ls=0.0200 drop=0.0068
  epoch 5/8  score=0.95214  best=0.95214 